In [ ]:
#| default_exp apkg

# Import `.apkg`

> Load an Anki package (`.apkg`) into a collection, matching notes by guid

An `.apkg` is how decks are usually exchanged: a zip holding a schema 11 collection and the media files. Libraries such as [genanki](https://github.com/kerrickstaley/genanki) write them, so a deck can be built from text files and published to AnkiWeb without a desktop Anki: import the package here, then `sync`.

Anki itself imports an `.apkg` by matching notes on their **guid**, and sync matches them on their **id**. The two don't agree: a package rebuilt by genanki gives every note a fresh id but the same guid. So `import_apkg` matches on guid. A note that is already in the collection is updated in place, keeping its id, cards and review history, and a new note gets a new id. Importing the same package twice changes nothing.

What happens to each part of the package:

- **Notetypes** are matched by id. A new one is created. An existing one is replaced only if its css, fields or templates differ.
- **Decks** are matched by id. A new one is created, and an existing one is renamed if its name differs.
- **Notes** are matched by guid. Fields and tags are overwritten when they differ.
- **Cards** are matched by (note, template). A missing card is created as a new card. An existing card keeps its scheduling and is only moved if the package puts it in another deck (cards in a filtered deck are left alone).
- **Media** files are copied with `add_media`.
- Nothing is deleted: a note that is not in the package stays in the collection.

Adding, removing or reordering the fields or templates of a notetype is a schema change. Anki treats it as one and forces a full sync; the server refuses it as a delta. `import_apkg` bumps `scm` for such a change, so the next `sync` raises a full sync instead of failing. A full upload replaces the server copy, so `sync` first, to pull in reviews done on other devices, then import, then publish with `sync(upload=True)`. A bare `sync()` after such an import would download the server's copy over it.

In [ ]:
#| export
import json, sqlite3, tempfile, zipfile
from fastcore.utils import *
from fastanki.schema import *
from fastanki.collection import *
from fastanki.syncer import nt_from_s11, nt_to_s11, deck_from_s11
from fastanki.media import *

In [ ]:
import json
from fastcore.test import *

## Notetypes and decks

The package stores them as the schema 11 dicts that `syncer.py` already converts, so the existing converters do the work. `_upsert_model` returns whether anything changed and whether the structure did.

In [ ]:
#| export
def _structure(m): return [f['name'] for f in m['flds']], [t['name'] for t in m['tmpls']]

def _upsert_model(col, m, now_s):
    "Create notetype `m` (schema 11 dict) or replace it if it differs; returns (`'new'`|`'changed'`|`'same'`, structure changed)"
    m = dict(m, usn=-1, mod=now_s)
    row,flds,tmpls = nt_from_s11(m)
    ntid = row[0]
    if not col.q1('select 1 from notetypes where id=?', ntid): status,schema = 'new',False
    else:
        old = nt_to_s11(col.con, ntid)
        same = lambda d: (d['name'], d['css'], d['sortf'], d['type'], [(t['name'],t['qfmt'],t['afmt']) for t in d['tmpls']], _structure(d)[0])
        if same(old)==same(m): return 'same',False
        col.con.execute('delete from fields where ntid=?', (ntid,))
        col.con.execute('delete from templates where ntid=?', (ntid,))
        status,schema = 'changed',_structure(old)!=_structure(m)
    col.con.execute('insert or replace into notetypes values (?,?,?,?,?)', row)
    col.con.executemany('insert into fields values (?,?,?,?)', flds)
    col.con.executemany('insert into templates values (?,?,?,?,?,?)', tmpls)
    return status,schema

In [ ]:
#| export
def _upsert_deck(col, d, now_s):
    """Create deck `d` (schema 11 dict), rename it, or adopt the local deck of the same name.
    Returns (`'new'`|`'renamed'`|`'matched'`|`None`, local deck id). Anki's own importer gives decks new ids,
    so a collection filled by the app holds the package's decks by name under other ids; deck names are unique."""
    did,name = int(d['id']), d['name'].replace('::','\x1f')
    cur = col.q1('select name from decks where id=?', did)
    if cur is None:
        if other := col.q1('select id from decks where name=?', name): return 'matched',other
        col.con.execute('insert into decks values (?,?,?,?,?,?)', deck_from_s11(dict(d, usn=-1, mod=now_s)))
        return 'new',did
    if did!=1 and cur!=name and not col.q1('select 1 from decks where name=? and id!=?', name, did):
        col.con.execute('update decks set name=?, mtime_secs=?, usn=-1 where id=?', (name, now_s, did))
        return 'renamed',did
    return None,did

## Notes and cards

Note fields are copied as the package wrote them. The sort field and checksum are recomputed with the same functions `add` uses, since a package built by another library may compute them differently.

In [ ]:
#| export
@patch
def import_apkg(self:Collection, apkg):
    "Import the genanki-style `.apkg` at `apkg`, matching notes by guid; returns a dict of counts"
    st = dict(notes_new=0, notes_updated=0, notes_same=0, cards_new=0, cards_moved=0, models_new=0, models_changed=0,
              decks_new=0, decks_renamed=0, decks_matched=0, media=0, schema_changed=False)
    now_s = now_ms()//1000
    with tempfile.TemporaryDirectory() as tmp, zipfile.ZipFile(apkg) as z:
        names = z.namelist()
        if 'collection.anki21b' in names and 'collection.anki2' not in names:
            raise ValueError('modern (zstd) .apkg not supported: expected a genanki-style package')
        z.extract('collection.anki2', tmp)
        src = sqlite3.connect(f'file:{tmp}/collection.anki2?mode=ro', uri=True)
        models,decks = (json.loads(r) for r in src.execute('select models, decks from col').fetchone())
        notes = src.execute('select id, guid, mid, tags, flds from notes').fetchall()
        cards = src.execute('select nid, ord, did from cards').fetchall()
        src.close()
        # media first, so a note never cites a file that isn't there
        for num,fname in json.loads(z.read('media')).items():
            got = self.add_media(z.read(num), fname)
            if got!=fname: raise ValueError(f'media {fname!r} was stored as {got!r}; note fields cite the original name')
            st['media'] += 1
    by_note = {}
    for nid,ord_,did in cards: by_note.setdefault(nid, []).append((ord_,did))
    with self._tx():
        for m in models.values():
            status,schema = _upsert_model(self, m, now_s)
            if status!='same': st[f'models_{status}'] += 1
            st['schema_changed'] |= schema
        dmap = {}
        for d in sorted(decks.values(), key=lambda d: d['name'].count('::')):  # parents before children
            r,dmap[int(d['id'])] = _upsert_deck(self, d, now_s)
            if r: st[f'decks_{r}'] += 1
        for src_id,guid,mid,tags,flds in notes:
            vals = flds.split('\x1f')
            sfld,csum = strip_html_media(vals[int(models[str(mid)]['sortf'])]), field_csum(vals[0])
            row = self.con.execute('select id, flds, tags from notes where guid=?', (guid,)).fetchone()
            if not row:
                nid = ts_id(self.con, 'notes')
                self.con.execute("insert into notes values (?,?,?,?,-1,?,?,?,?,0,'')", (nid, guid, mid, now_s, tags, flds, sfld, csum))
                st['notes_new'] += 1
            else:
                nid = row[0]
                if (row[1],row[2])==(flds,tags): st['notes_same'] += 1
                else:
                    self.con.execute('update notes set flds=?, tags=?, sfld=?, csum=?, mod=?, usn=-1 where id=?', (flds, tags, sfld, csum, now_s, nid))
                    st['notes_updated'] += 1
            for ord_,did in by_note.get(src_id, []):
                did = dmap.get(did, did)
                cur = self.con.execute('select id, did, odid from cards where nid=? and ord=?', (nid, ord_)).fetchone()
                if cur:
                    if cur[1]!=did and not cur[2]:  # odid set: card sits in a filtered deck, leave it
                        self.con.execute('update cards set did=?, mod=?, usn=-1 where id=?', (did, now_s, cur[0]))
                        st['cards_moved'] += 1
                else:
                    self.con.execute("insert into cards values (?,?,?,?,?,-1,0,0,?,0,0,0,0,0,0,0,0,'{}')",
                        (ts_id(self.con, 'cards'), nid, did, ord_, now_s, self._next_pos()))
                    st['cards_new'] += 1
        # a structure change must force a full sync (Anki's own rule); bumping `mod` alone is not enough, see DEV.md
        if st['schema_changed']: self.con.execute('update col set scm=?', (now_ms(),))
        self._dirty()
    return st

## Example

A package with a two-card notetype and a cloze, built with genanki.

In [ ]:
import genanki
QR = genanki.Model(1607392319, 'QR', fields=[{'name':'Q'}, {'name':'A'}], templates=[
    {'name':'fwd', 'qfmt':'{{Q}}', 'afmt':'{{FrontSide}}<hr id=answer>{{A}}'}])

def build(path, a='first answer', tags=('t',), deck='Maths::Test', did=2059400110):
    d = genanki.Deck(did, deck)
    d.add_note(genanki.Note(QR, ['What is $x^2$?', a], tags=list(tags), guid='guid-1'))
    genanki.Package(d).write_to_file(str(path))
    return path

In [ ]:
tmp = Path(tempfile.mkdtemp())
col = Collection.open(tmp/'c.anki2')
col.import_apkg(build(tmp/'1.apkg'))

Importing the same content again, even from a rebuilt package with new ids, changes nothing:

In [ ]:
st = col.import_apkg(build(tmp/'2.apkg'))
test_eq((st['notes_new'], st['notes_same']), (0,1))
test_eq(col.q1('select count(*) from notes'), 1)

A changed note is updated in place. The note keeps its id, so its cards and reviews stay attached:

In [ ]:
nid = col.q1("select id from notes where guid='guid-1'")
st = col.import_apkg(build(tmp/'3.apkg', a='second answer', tags=('t','edited')))
test_eq(st['notes_updated'], 1)
test_eq(col.q1("select id from notes where guid='guid-1'"), nid)
col.get_note(nid)

Renaming a deck or moving a card keeps its scheduling:

In [ ]:
st = col.import_apkg(build(tmp/'4.apkg', a='second answer', tags=('t','edited'), deck='Maths::Renamed'))
test_eq(st['decks_renamed'], 1)
st = col.import_apkg(build(tmp/'5.apkg', a='second answer', tags=('t','edited'), deck='Other', did=2059400111))
test_eq((st['decks_new'], st['cards_moved'], st['schema_changed']), (1,1,False))